# Nepali news classification: train on Colab

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. You need the project files and `data/news.csv` (see the README on how to create it from np20ng).

The full run took about 2.5 minutes per epoch on a T4.

In [ ]:
!nvidia-smi -L

In [ ]:
# Get the code (replace with your fork if you have one)
!git clone https://github.com/PrishniGhimire/nepali-news-classification.git
%cd nepali-news-classification
!pip install -q -r requirements.txt

## Add the data

Upload `news.csv` (the 10,000-article sample) into the `data/` folder using the Files panel on the left, or run the next cell to pick the file.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('data', exist_ok=True)
up = files.upload()          # choose news.csv
for name in up:
    shutil.move(name, 'data/news.csv')
!ls -la data

In [ ]:
!python -m pytest -q

In [ ]:
# Quick check that training works (tiny subset, scores are NOT real results)
!python -m src.train_transformer --config configs/xlm-roberta-base.json --smoke_test

In [ ]:
# Baseline
!python -m src.baseline --data data/news.csv --output_dir outputs/baseline

In [ ]:
# Full fine-tuning run
!python -m src.train_transformer --config configs/xlm-roberta-base.json

In [ ]:
!python -m src.compare
!python -m src.inspect_data --data data/news.csv --tokenizer xlm-roberta-base
!python -m src.error_analysis --predictions outputs/xlm-roberta-base/test_predictions.csv --title xlm-roberta-base --other outputs/baseline/test_predictions.csv --other_title baseline

## Download results

This zips only the small result files (not the 1 GB model).

In [ ]:
!zip -r results_download.zip outputs/baseline outputs/xlm-roberta-base results -x '*/best/*' '*.joblib'
from google.colab import files
files.download('results_download.zip')